# 【番外編】Google Colab + AutoGluon で機械学習を体験する

**テーマ**：健康診断データから「糖尿病かどうか」を予測する  
**目標**：複数の機械学習モデルを一気に比較し、最良モデルで予測を体験する

---
このノートブックは「AIのしくみを感覚的に理解する」シリーズの【寄り道】記事に対応しています。  
上から順にセルを実行（Shift+Enter）していくだけで体験できます。

**注意**：Google Colabはセッションを開くたびにインストールが必要です。毎回ステップ①から実行してください。

## ステップ① AutoGluonのインストール

AutoGluon は Google Colab には入っていないため、最初に1度だけインストールします。  
少し時間がかかります（3〜5分程度）。エラーのような表示が出ても最後まで実行されれば問題ありません。

In [ ]:
# AutoGluonのインストール（表形式データ用）
!pip install autogluon.tabular --quiet

## ステップ② データの読み込み

今回使うのは**ピマ・インディアン糖尿病データセット**です。  
米国国立糖尿病・消化器・腎臓病研究所が収集した実際の健康診断データで、  
768人分の検査値から「糖尿病かどうか」を予測します。

| 列名 | 意味 | 単位 |
|------|------|------|
| Pregnancies | 妊娠回数 | 回 |
| Glucose | 血糖値 | mg/dL |
| BloodPressure | 拡張期血圧 | mmHg |
| SkinThickness | 皮膚の厚さ（体脂肪の指標） | mm |
| Insulin | インスリン値 | μU/mL |
| BMI | 体格指数（肥満度） | kg/m² |
| DiabetesPedigreeFunction | 糖尿病遺伝リスク指数 | ― |
| Age | 年齢 | 歳 |
| **Outcome** | **糖尿病かどうか（予測対象）** | **1=あり、0=なし** |

In [ ]:
from autogluon.tabular import TabularDataset, TabularPredictor

# データをURLから直接読み込む
url = 'https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv'
data = TabularDataset(url)

print(f'データ件数: {len(data)}件, 列数: {len(data.columns)}列')
data.head()

## ステップ③ データの確認

学習を始める前に、データの中身を確認しておきましょう。

In [ ]:
# 糖尿病あり/なしの内訳
print('Outcomeの内訳:')
print(data['Outcome'].value_counts())
print(f'\n糖尿病あり: {data["Outcome"].sum()}人 ({data["Outcome"].mean()*100:.1f}%)')
print(f'糖尿病なし: {(data["Outcome"]==0).sum()}人 ({(1-data["Outcome"].mean())*100:.1f}%)')

## ステップ④ 特徴量の分布を可視化する

学習の前に「どの特徴量が糖尿病と関係しているか」を目で確認してみましょう。  
血糖値（Glucose）・BMI・血圧（BloodPressure）の3つを比較します。

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

features = [
    ('Glucose', 'Glucose (Blood Sugar)'),
    ('BMI', 'BMI'),
    ('BloodPressure', 'Blood Pressure')
]

for ax, (col, xlabel) in zip(axes, features):
    for outcome, label, color in [(0, 'No Diabetes', 'blue'), (1, 'Diabetes', 'red')]:
        subset = data[data['Outcome'] == outcome][col]
        ax.hist(subset, bins=20, alpha=0.6, label=label, color=color)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Count')
    ax.set_title(col)
    ax.legend()

plt.suptitle('Feature Distribution by Diabetes Outcome', y=1.02)
plt.tight_layout()
plt.show()

### グラフの読み方

- **Glucose**：青と赤の分布が大きくずれている → 糖尿病との関係が強い
- **BMI**：赤が右寄り（高BMI側）に多い → ある程度の関係がある
- **Blood Pressure**：青と赤がほぼ同じ分布 → このデータでは関係が見えにくい

---
**⚠️ 重要な注意**  
「血圧の分布がほぼ重なる」という結果は、このデータセット上の統計的な傾向です。  
医学的には血圧と糖尿病は関連していることが知られており、  
「血圧と糖尿病は無関係」という医学的な結論ではありません。  
データが少ない・特定の集団に限定されているなど、データ自体の限界から来ている可能性があります。  
**健康に関する判断は必ず医師にご相談ください。**

## ステップ⑤ 学習データと検証データに分割

データを「学習に使う分（80%）」と「精度を確認するための分（20%）」に分けます。  
検証データはモデルが「見たことのないデータ」なので、実際の性能を測るのに使います。

**なぜ分けるのか？**  
学習に使ったデータで精度を測ると、「答えを覚えたテスト」になってしまうからです。  
これを過学習と言います（第3回で説明しました）。

In [ ]:
# 学習データ（80%）と検証データ（20%）に分割
train_data = data.sample(frac=0.8, random_state=42)
test_data  = data.drop(train_data.index)

print(f'学習データ: {len(train_data)}件')
print(f'検証データ: {len(test_data)}件')

## ステップ⑥ モデルの学習と比較

いよいよ学習です。第1〜8回で学んだモデルたちを一気に比較してみましょう。

### 今回使うモデル
| キー | モデル名 | シリーズの回 |
|------|---------|------------|
| LR | ロジスティック回帰 | 第4回 |
| RF | ランダムフォレスト | 第6回 |
| XT | Extra Trees | 第6回コラム |
| GBM | LightGBM | 第8回 |
| XGB | XGBoost | 第8回 |

**※ 注意**：SVMと決定木はAutoGluonに標準搭載されていません。  
SVMはデータ件数が増えると計算量が急増するため、スケーラビリティの観点から除外されています。

### 主なパラメータ
| パラメータ | 意味 | 今回の設定 |
|-----------|------|----------|
| `label` | 予測したい列名 | `'Outcome'` |
| `eval_metric` | 評価指標 | `'roc_auc'`（AUC） |
| `presets` | 速度と精度のバランス | `'medium_quality'` |

In [ ]:
# 使用するモデルを指定
models = {
    'LR' : {},   # ロジスティック回帰
    'RF' : {},   # ランダムフォレスト
    'XT' : {},   # Extra Trees
    'GBM': {},   # LightGBM
    'XGB': {},   # XGBoost
}

# モデルの学習（複数モデルを自動で比較）
predictor = TabularPredictor(
    label='Outcome',
    eval_metric='roc_auc'
).fit(
    train_data,
    hyperparameters=models,
    presets='medium_quality'
)

## ステップ⑦ モデル比較表の確認

学習が完了したら、モデルの比較表を表示してみましょう。

### 精度指標の読み方
| 指標 | 意味 |
|------|------|
| **score_val** | 検証データでのAUCスコア（1.0が最高、0.5はランダムと同じ） |
| **fit_time** | 学習にかかった時間（秒） |

一番下の**WeightedEnsemble_L2**はAutoGluonが自動で作るアンサンブルモデルです。  
複数のモデルを組み合わせることでさらに精度を高めています（アンサンブルについては後述）。

In [ ]:
# モデル比較表の表示
leaderboard = predictor.leaderboard(silent=True)
print(leaderboard[['model', 'score_val', 'fit_time']].to_string())

## ステップ⑧ 特徴量重要度の確認

「どの検査値が予測に最も貢献しているか」を確認します。  
ステップ④で目で見た傾向と一致しているでしょうか？

In [ ]:
# 特徴量重要度の計算とグラフ表示
importance = predictor.feature_importance(test_data)

plt.figure(figsize=(8, 5))
importance['importance'].sort_values().plot(
    kind='barh', color='steelblue'
)
plt.xlabel('Importance Score')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

## ステップ⑨ ROC曲線の描画

ROC曲線は、モデルの「識別力」を視覚的に表したグラフです。  
曲線が左上に近いほど性能が高く、AUCはその面積を表します。  
AUC=1.0が完璧、AUC=0.5はランダムと同じです。

In [ ]:
from sklearn.metrics import roc_curve, auc

# 予測確率の取得
proba   = predictor.predict_proba(test_data)
y_true  = test_data['Outcome']
y_score = proba[1]  # 糖尿病あり（1）の確率

# ROC曲線の描画
fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'AUC = {roc_auc:.3f}')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random (AUC=0.5)')
plt.xlabel('False Positive Rate (Healthy predicted as Diabetic)')
plt.ylabel('True Positive Rate (Diabetic correctly identified)')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

## ステップ⑩ 新しいデータで予測してみる

架空の検査値を入力して予測してみましょう。数値を自由に変えて試してみてください。

In [ ]:
import pandas as pd

# 架空の患者データ（自由に数値を変えて試してみてください）
new_patients = pd.DataFrame({
    'Pregnancies'             : [1,    5,    0   ],
    'Glucose'                 : [85,   168,  120 ],
    'BloodPressure'           : [66,   74,   70  ],
    'SkinThickness'           : [29,   35,   20  ],
    'Insulin'                 : [0,    168,  85  ],
    'BMI'                     : [26.6, 43.1, 30.0],
    'DiabetesPedigreeFunction': [0.351, 2.288, 0.500],
    'Age'                     : [31,   33,   45  ]
})

# 予測の実行
predictions = predictor.predict(new_patients)
proba_new   = predictor.predict_proba(new_patients)

# 結果の表示
result = new_patients.copy()
result['予測'] = predictions.map({1: '糖尿病あり', 0: '糖尿病なし'})
result['確信度'] = proba_new[1].round(3)
print(result[['Glucose', 'BMI', 'Age', '予測', '確信度']].to_string())

## ステップ⑪ 自分のCSVファイルを使う場合

今回はURLからデータを読み込みましたが、自分のデータを使う場合は  
CSVファイルをGoogle Driveにアップロードして読み込むことができます。

In [ ]:
# ── 自分のデータを使う場合（参考コード）──

# ① Google Driveをマウントする
# from google.colab import drive
# drive.mount('/content/drive')

# ② CSVファイルを読み込む
# my_data = TabularDataset('/content/drive/MyDrive/your_file.csv')

# ③ あとは同じ手順で学習・予測できる
# predictor = TabularPredictor(label='目的変数の列名').fit(my_data)

print('自分のデータを使う場合は、上のコードのコメントを外して実行してください。')

## まとめ

AutoGluonを使うことで、次のことがわずか数行のコードで実現できました：

1. **データの読み込み**：URLから直接読み込み
2. **前処理**：欠損値の補完・型変換を自動処理
3. **モデル比較**：5つのモデルを一気に学習・評価
4. **可視化**：特徴量の分布・重要度・ROC曲線
5. **予測**：新しいデータへの適用

---

## データ分析をする上での注意点

### ① モデルの精度が高い ≠ 良い分析
何を予測すべきか、なぜそれを知りたいのかという問いを立てるのは人間です。  
精度の高いモデルを作ることがゴールではなく、そこから何を読み取るかが大切です。

### ② 特徴量の選択は人間の仮説
「血糖値やBMIが糖尿病に関係しているはず」という仮説があってこそ特徴量として選べます。  
モデルはその仮説を検証する道具です。仮説なき分析は意味のある結果を生みません。

### ③ データが語ることと、現実は違うことがある
今回、血圧（BloodPressure）のヒストグラムは糖尿病あり/なしでほぼ同じ分布を示しました。  
しかしこれは「血圧と糖尿病は無関係」という意味ではありません。  
このデータはピマ・インディアン女性768人という特定の集団に限定されており、  
データ件数が少ない・集団の偏りがあるなど、**データ自体の限界**から来ている可能性があります。  
「データがそう言っているから正しい」という短絡的な解釈は危険です。  
医学的には血圧・肥満・血糖値はいずれも糖尿病と深く関連していることが知られています。

⚠️ **健康に関する判断は必ず医師にご相談ください。このノートブックの結果は医学的な診断の代替にはなりません。**

### ④ 結果をどう使うかの判断は人間にある
予測結果をもとに何を決断するかは人間の責任です。  
特に医療・福祉・金融など人の生活に関わる分野では、モデルの出力を鵜呑みにせず慎重に判断することが求められます。

### ⑤ ブラックボックスへの向き合い方
AutoGluonで一発比較できても、第1〜8回で学んだ「中身の理解」があってこそ結果を正しく解釈できます。  
「なぜこのモデルが強いのか」「この特徴量がなぜ重要なのか」を考える姿勢が大切です。

---

## アンサンブルについて

比較表の一番上に**WeightedEnsemble_L2**というモデルが表示されています。  
これはAutoGluonが自動で作る「アンサンブルモデル」です。

アンサンブルとは複数のモデルの予測を組み合わせて、単体より精度を高める手法です。  
（詳しくは第6回のランダムフォレストの説明を参照してください）

AutoGluonはこの処理を自動で行い、最も精度の高い組み合わせを見つけてくれます。  
今回の結果では、LinearModel（ロジスティック回帰）とRandomForestを組み合わせていました。